# 03 — Network Analysis
## SMNA2026 Assignment 2: Public Sentiment Toward Electric Vehicles in Australia

**Team:** Ansh Anand Parekh (s4060237), Disha Dogra (s4091900), Syna Arora (s4109652)

### What this notebook does:
1. Constructs a directed reply graph from Reddit EV comments
2. Computes SNA measures: degree, eigenvector, Katz centrality, betweenness, closeness
3. Analyses graph structure: clustering, connected components, bridges
4. Detects communities using the Louvain method
5. Visualises the network and key findings
6. Saves the graph as GraphML for further analysis

### Network Definition:
- **Nodes** = Reddit users who commented on EV-related posts
- **Edges** = Directed reply relationships (user A replied to user B)
- **Direction** = A → B means A replied to B
- **Weight** = Number of times A replied to B

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import warnings
warnings.filterwarnings('ignore')

# Install and import community detection
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', '-q'])
import community as community_louvain

PROCESSED_DIR = '../data/processed/'
FIGURES_DIR   = '../reports/figures/'
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Libraries imported!')
print(f'NetworkX version : {nx.__version__}')

Libraries imported!
NetworkX version : 3.5



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


## 2. Load Processed Data

In [2]:
posts_df    = pd.read_csv(os.path.join(PROCESSED_DIR, 'posts_clean.csv'))
comments_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'comments_clean.csv'))

print(f'Posts loaded    : {len(posts_df)}')
print(f'Comments loaded : {len(comments_df)}')
print(f'Unique authors  : {comments_df["author"].nunique()}')
print(f'\nComments by subreddit:')
print(comments_df['subreddit'].value_counts())

Posts loaded    : 311
Comments loaded : 84281
Unique authors  : 23561

Comments by subreddit:
subreddit
electricvehicles    81899
AusFinance           1938
australia             444
Name: count, dtype: int64


## 3. Construct Reply Graph

We construct a directed reply graph where:
- Each **node** is a Reddit user with attributes (comment count, score)
- Each **edge** A → B means user A replied to user B
- Edge **weight** = number of times A replied to B

The `parent_id` field tells us who each comment is replying to:
- `t3_xxx` = reply to the original post (top level comment)
- `t1_xxx` = reply to another comment (creates a network edge)

In [3]:
# Build lookup: comment_id -> author
comment_author_lookup = comments_df.set_index('comment_id')['author_anon'].to_dict()

# Build lookup: author -> total score and comment count
author_stats = comments_df.groupby('author_anon').agg(
    comment_count=('comment_id', 'count'),
    total_score=('score', 'sum')
).to_dict('index')

# Construct directed reply graph
replyGraph = nx.DiGraph()

edges_added  = 0
nodes_added  = 0
skipped      = 0

for _, row in comments_df.iterrows():
    author    = row['author_anon']
    parent_id = str(row.get('parent_id', ''))

    # Skip anonymous/deleted users
    if author == 'anonymous':
        skipped += 1
        continue

    # Add node with attributes if not already present
    if author not in replyGraph:
        stats = author_stats.get(author, {})
        replyGraph.add_node(
            author,
            comment_count = stats.get('comment_count', 0),
            total_score   = stats.get('total_score', 0)
        )
        nodes_added += 1

    # Only create edges for replies to other comments (t1_ prefix)
    if parent_id.startswith('t1_'):
        parent_comment_id = parent_id[3:]
        parent_author     = comment_author_lookup.get(parent_comment_id)

        if parent_author and parent_author != 'anonymous' and parent_author != author:
            # Add parent node if not present
            if parent_author not in replyGraph:
                stats = author_stats.get(parent_author, {})
                replyGraph.add_node(
                    parent_author,
                    comment_count = stats.get('comment_count', 0),
                    total_score   = stats.get('total_score', 0)
                )

            # Add or update edge weight
            if replyGraph.has_edge(author, parent_author):
                replyGraph[author][parent_author]['weight'] += 1
            else:
                replyGraph.add_edge(author, parent_author, weight=1)
            edges_added += 1

print('Reply Graph Constructed!')
print(f'  Nodes (users)  : {replyGraph.number_of_nodes():,}')
print(f'  Edges (replies): {replyGraph.number_of_edges():,}')
print(f'  Skipped (anon) : {skipped:,}')

Reply Graph Constructed!
  Nodes (users)  : 23,561
  Edges (replies): 47,860
  Skipped (anon) : 0


## 4. Basic Graph Statistics

In [4]:
print('Computing basic graph statistics...')
print('=' * 55)

N = replyGraph.number_of_nodes()
M = replyGraph.number_of_edges()

# Density
density = nx.density(replyGraph)

# Transitivity (global clustering coefficient)
transitivity = nx.transitivity(replyGraph)

# Average clustering coefficient
avg_clustering = nx.average_clustering(replyGraph.to_undirected())

# Connected components
num_wcc = nx.number_weakly_connected_components(replyGraph)
num_scc = nx.number_strongly_connected_components(replyGraph)

# Largest weakly connected component
largest_wcc      = max(nx.weakly_connected_components(replyGraph), key=len)
largest_wcc_size = len(largest_wcc)

# Reciprocity
reciprocity = nx.overall_reciprocity(replyGraph)

# Degree stats
in_degrees  = dict(replyGraph.in_degree())
out_degrees = dict(replyGraph.out_degree())
avg_in_deg  = np.mean(list(in_degrees.values()))
avg_out_deg = np.mean(list(out_degrees.values()))

print(f'GRAPH STRUCTURE')
print(f'  Nodes (users)              : {N:,}')
print(f'  Edges (reply links)        : {M:,}')
print(f'  Network density            : {density:.6f}')
print(f'  Graph type                 : Directed, Weighted')
print(f'')
print(f'CONNECTIVITY')
print(f'  Weakly connected components: {num_wcc:,}')
print(f'  Strongly connected comps   : {num_scc:,}')
print(f'  Largest WCC size           : {largest_wcc_size:,} ({largest_wcc_size/N*100:.1f}% of nodes)')
print(f'')
print(f'INTERACTION PATTERNS')
print(f'  Global clustering coeff    : {transitivity:.4f}')
print(f'  Avg local clustering coeff : {avg_clustering:.4f}')
print(f'  Reciprocity                : {reciprocity:.4f}')
print(f'')
print(f'DEGREE')
print(f'  Avg in-degree              : {avg_in_deg:.2f}')
print(f'  Avg out-degree             : {avg_out_deg:.2f}')
print(f'  Max in-degree              : {max(in_degrees.values()):,}')
print(f'  Max out-degree             : {max(out_degrees.values()):,}')

Computing basic graph statistics...
GRAPH STRUCTURE
  Nodes (users)              : 23,561
  Edges (reply links)        : 47,860
  Network density            : 0.000086
  Graph type                 : Directed, Weighted

CONNECTIVITY
  Weakly connected components: 7,356
  Strongly connected comps   : 15,073
  Largest WCC size           : 15,891 (67.4% of nodes)

INTERACTION PATTERNS
  Global clustering coeff    : 0.0119
  Avg local clustering coeff : 0.0203
  Reciprocity                : 0.3725

DEGREE
  Avg in-degree              : 2.03
  Avg out-degree             : 2.03
  Max in-degree              : 313
  Max out-degree             : 448


## 5. Centrality Measures

We compute the centrality measures taught in class:
- **Degree centrality**: Who connects with the most users
- **Eigenvector centrality**: Who connects with other important users
- **Katz centrality**: Extended eigenvector considering all paths
- **Betweenness centrality**: Who bridges different groups
- **Closeness centrality**: Who can reach others fastest
- **PageRank**: Overall influence weighted by connection quality

In [6]:
print('Computing centrality measures...')

# 1. Degree centrality — works on full graph
lDegCentrality    = nx.degree_centrality(replyGraph)
lInDegCentrality  = nx.in_degree_centrality(replyGraph)
lOutDegCentrality = nx.out_degree_centrality(replyGraph)
print('Degree centrality done')

# Get largest weakly connected component
# Eigenvector and Katz don't work on disconnected graphs
largest_wcc = max(nx.weakly_connected_components(replyGraph), key=len)
G_lcc       = replyGraph.subgraph(largest_wcc).copy()
print(f'Largest component: {G_lcc.number_of_nodes()} nodes '
      f'({G_lcc.number_of_nodes()/replyGraph.number_of_nodes()*100:.1f}% of graph)')

# 2. Eigenvector centrality on largest component
try:
    lEigenVectorCentrality = nx.eigenvector_centrality_numpy(G_lcc, weight='weight')
except Exception as e:
    print(f'Eigenvector numpy failed: {e}')
    lEigenVectorCentrality = nx.eigenvector_centrality(G_lcc, max_iter=1000)
lEigenVectorCentrality = {n: lEigenVectorCentrality.get(n, 0) for n in replyGraph.nodes()}
print('Eigenvector centrality done')

# 3. Katz centrality on largest component
try:
    lKatzCentrality = nx.katz_centrality_numpy(G_lcc, weight='weight')
except Exception as e:
    print(f'Katz numpy failed: {e}')
    lKatzCentrality = nx.katz_centrality(G_lcc, max_iter=1000)
lKatzCentrality = {n: lKatzCentrality.get(n, 0) for n in replyGraph.nodes()}
print('Katz centrality done')

# 4. Betweenness centrality
if N > 5000:
    print('Large network — using sampled betweenness (k=500)...')
    lBetweennessCentrality = nx.betweenness_centrality(replyGraph, k=500, normalized=True)
else:
    lBetweennessCentrality = nx.betweenness_centrality(replyGraph, normalized=True)
print('Betweenness centrality done')

# 5. Closeness centrality
lClosenessCentrality = nx.closeness_centrality(replyGraph)
print('Closeness centrality done')

# 6. PageRank — works on full graph
lPageRank = nx.pagerank(replyGraph, weight='weight', alpha=0.85)
print('PageRank done')

# Add centrality values as node attributes
for nodeId, cent in lEigenVectorCentrality.items():
    replyGraph.nodes[nodeId]['eigen'] = float(cent)
for nodeId, cent in lKatzCentrality.items():
    replyGraph.nodes[nodeId]['katz'] = float(cent)
for nodeId, cent in lPageRank.items():
    replyGraph.nodes[nodeId]['pagerank'] = float(cent)
for nodeId, cent in lBetweennessCentrality.items():
    replyGraph.nodes[nodeId]['betweenness'] = float(cent)

print('\nAll centrality measures computed and added as node attributes!')

Computing centrality measures...
Degree centrality done
Largest component: 15891 nodes (67.4% of graph)
Eigenvector numpy failed: `eigenvector_centrality_numpy` does not give consistent results for disconnected graphs
Eigenvector centrality done
Katz centrality done
Large network — using sampled betweenness (k=500)...
Betweenness centrality done
Closeness centrality done
PageRank done

All centrality measures computed and added as node attributes!


In [ ]:
# Build centrality DataFrame for analysis
centrality_df = pd.DataFrame({
    'user'         : list(replyGraph.nodes()),
    'degree'       : [lDegCentrality.get(n, 0)         for n in replyGraph.nodes()],
    'in_degree'    : [lInDegCentrality.get(n, 0)       for n in replyGraph.nodes()],
    'out_degree'   : [lOutDegCentrality.get(n, 0)      for n in replyGraph.nodes()],
    'eigenvector'  : [lEigenVectorCentrality.get(n, 0) for n in replyGraph.nodes()],
    'katz'         : [lKatzCentrality.get(n, 0)        for n in replyGraph.nodes()],
    'betweenness'  : [lBetweennessCentrality.get(n, 0) for n in replyGraph.nodes()],
    'closeness'    : [lClosenessCentrality.get(n, 0)   for n in replyGraph.nodes()],
    'pagerank'     : [lPageRank.get(n, 0)              for n in replyGraph.nodes()],
    'in_deg_count' : [in_degrees.get(n, 0)             for n in replyGraph.nodes()],
    'out_deg_count': [out_degrees.get(n, 0)            for n in replyGraph.nodes()]
})

centrality_df = centrality_df.sort_values('pagerank', ascending=False).reset_index(drop=True)

print('Top 15 Most Influential Users (PageRank):')
print('-' * 75)
print(f'{"Rank":<5}{"User":<15}{"PageRank":<12}{"Eigenvector":<14}{"Katz":<12}{"In-Deg":<8}')
print('-' * 75)
for i, row in centrality_df.head(15).iterrows():
    print(f'{i+1:<5}{row["user"]:<15}{row["pagerank"]:.6f}   {row["eigenvector"]:.6f}     {row["katz"]:.6f}  {row["in_deg_count"]:<8}')

# Save centrality results
centrality_df.to_csv(os.path.join(PROCESSED_DIR, 'centrality_scores.csv'), index=False)
print('\nCentrality scores saved!')

## 6. Centrality Distribution Plots

In [ ]:
# Plot centrality distributions as taught in class
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

centrality_measures = [
    ('degree',      lDegCentrality,         'Degree Centrality',      'steelblue'),
    ('eigenvector', lEigenVectorCentrality,  'Eigenvector Centrality', 'coral'),
    ('katz',        lKatzCentrality,         'Katz Centrality',        'purple'),
    ('betweenness', lBetweennessCentrality,  'Betweenness Centrality', 'green'),
    ('closeness',   lClosenessCentrality,    'Closeness Centrality',   'orange'),
    ('pagerank',    lPageRank,               'PageRank',               'darkred')
]

for ax, (col, centrality_dict, title, colour) in zip(axes.flatten(), centrality_measures):
    values = list(centrality_dict.values())
    ax.hist(values, bins=50, color=colour, alpha=0.7, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Centrality Score')
    ax.set_ylabel('Number of Users')
    ax.set_yscale('log')

plt.suptitle('Centrality Distributions — EV Reddit Reply Network', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'centrality_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Centrality distributions saved!')

## 7. Bridges Analysis

In [ ]:
# Bridges analysis — convert to undirected as taught in class
G_undirected = replyGraph.to_undirected()

print('Computing bridges...')
try:
    bridges     = list(nx.bridges(G_undirected))
    num_bridges = len(bridges)
    print(f'Number of bridges: {num_bridges:,}')
    if num_bridges > 0 and num_bridges <= 10:
        print('Bridge connections (first 10):')
        for u, v in bridges[:10]:
            print(f'  {u} -- {v}')
    elif num_bridges > 10:
        print(f'(showing first 5 of {num_bridges})')
        for u, v in bridges[:5]:
            print(f'  {u} -- {v}')
except Exception as e:
    print(f'Bridge calculation skipped: {e}')
    num_bridges = 0

print(f'\nGraph structure summary:')
print(f'  Weakly connected components  : {num_wcc:,}')
print(f'  Strongly connected components: {num_scc:,}')
print(f'  Bridges                      : {num_bridges:,}')
print(f'  Global clustering (transitivity): {transitivity:.4f}')
print(f'  Reciprocity                  : {reciprocity:.4f}')

## 8. Community Detection (Louvain)

In [ ]:
print('Detecting communities using Louvain algorithm...')

# Louvain works on undirected graph
partition       = community_louvain.best_partition(G_undirected, weight='weight')
num_communities = len(set(partition.values()))
modularity      = community_louvain.modularity(partition, G_undirected)

# Community sizes
community_sizes = pd.Series(partition.values()).value_counts().sort_values(ascending=False)

print(f'\nCommunity Detection Results:')
print(f'  Algorithm         : Louvain')
print(f'  Communities found : {num_communities}')
print(f'  Modularity score  : {modularity:.4f}')
print(f'  Largest community : {community_sizes.iloc[0]} users')
print(f'  Smallest community: {community_sizes.iloc[-1]} users')
print(f'  Average size      : {community_sizes.mean():.1f} users')
print(f'\n  Higher modularity (>0.3) = strong community structure')

print(f'\nTop 10 Communities by Size:')
print('-' * 45)
for rank, (comm_id, size) in enumerate(community_sizes.head(10).items()):
    comm_users = centrality_df[centrality_df['user'].isin(
        [u for u, c in partition.items() if c == comm_id]
    )]
    top_user = comm_users.nlargest(1, 'pagerank')['user'].values
    top_user = top_user[0] if len(top_user) > 0 else 'N/A'
    print(f'  Community {rank+1:>2}: {size:>5} users | Top influencer: {top_user}')

# Add community to node attributes
for nodeId, comm in partition.items():
    if nodeId in replyGraph.nodes:
        replyGraph.nodes[nodeId]['community'] = int(comm)

# Add community to centrality df
centrality_df['community'] = centrality_df['user'].map(partition)
centrality_df.to_csv(os.path.join(PROCESSED_DIR, 'centrality_scores.csv'), index=False)

# Save community assignments
community_df = pd.DataFrame([
    {'user': u, 'community': c} for u, c in partition.items()
])
community_df.to_csv(os.path.join(PROCESSED_DIR, 'communities.csv'), index=False)
print(f'\nCommunity assignments saved!')

## 9. Save Graph as GraphML

In [ ]:
# Save graph as GraphML — can be opened in Gephi for further visualisation
graphml_path     = os.path.join(PROCESSED_DIR, 'ev_reply_graph.graphml')
mod_graphml_path = os.path.join(PROCESSED_DIR, 'mod_ev_reply_graph.graphml')

# Basic graph
nx.write_graphml(replyGraph, graphml_path)
print(f'Graph saved: {graphml_path}')

# Modified graph with all centrality attributes
nx.write_graphml(replyGraph, mod_graphml_path)
print(f'Modified graph saved: {mod_graphml_path}')
print('(Can be opened in Gephi for advanced visualisation)')

## 10. Network Visualisation

In [ ]:
print('Creating network visualisation (top 150 users by PageRank)...')

# Subgraph of top 150 users for readability
top_users = centrality_df.head(150)['user'].tolist()
G_sub     = replyGraph.subgraph(top_users).copy()

# Community colours
sub_partition = {n: partition.get(n, 0) for n in G_sub.nodes()}
unique_comms  = sorted(set(sub_partition.values()))
cmap          = plt.cm.Set3
colours       = {c: cmap(i / max(len(unique_comms), 1)) for i, c in enumerate(unique_comms)}
node_colours  = [colours[sub_partition[n]] for n in G_sub.nodes()]

# Node sizes proportional to PageRank
pr_vals    = [lPageRank.get(n, 0) for n in G_sub.nodes()]
max_pr     = max(pr_vals) if pr_vals else 1
node_sizes = [max(30, (pr / max_pr) * 800) for pr in pr_vals]

# Spring layout
pos = nx.spring_layout(G_sub, k=2.0, seed=42, iterations=50)

fig, ax = plt.subplots(figsize=(16, 12))

nx.draw_networkx_edges(
    G_sub, pos, alpha=0.1, arrows=True,
    arrowsize=6, edge_color='grey', ax=ax
)
nx.draw_networkx_nodes(
    G_sub, pos, node_color=node_colours,
    node_size=node_sizes, alpha=0.85, ax=ax
)

# Label only top 15 users
top15  = centrality_df.head(15)['user'].tolist()
labels = {n: n for n in top15 if n in G_sub.nodes()}
nx.draw_networkx_labels(G_sub, pos, labels=labels, font_size=7, font_weight='bold', ax=ax)

# Legend for communities
legend_patches = [
    mpatches.Patch(color=colours[c], label=f'Community {i+1}')
    for i, c in enumerate(unique_comms[:8])
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=8, title='Communities')

ax.set_title(
    'EV Reddit Reply Network — Top 150 Users\n'
    'Node size = PageRank | Colour = Community | Edge = Reply',
    fontsize=13, fontweight='bold'
)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'network_graph.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Network visualisation saved!')

## 11. Top Users Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top 10 by PageRank
top10_pr = centrality_df.head(10)
axes[0,0].barh(range(len(top10_pr)), top10_pr['pagerank'], color='steelblue', alpha=0.8)
axes[0,0].set_yticks(range(len(top10_pr)))
axes[0,0].set_yticklabels(top10_pr['user'], fontsize=8)
axes[0,0].invert_yaxis()
axes[0,0].set_title('Top 10 — PageRank\n(Overall Influence)', fontweight='bold')
axes[0,0].set_xlabel('PageRank Score')

# Top 10 by Betweenness
top10_bw = centrality_df.nlargest(10, 'betweenness')
axes[0,1].barh(range(len(top10_bw)), top10_bw['betweenness'], color='coral', alpha=0.8)
axes[0,1].set_yticks(range(len(top10_bw)))
axes[0,1].set_yticklabels(top10_bw['user'], fontsize=8)
axes[0,1].invert_yaxis()
axes[0,1].set_title('Top 10 — Betweenness\n(Community Bridges)', fontweight='bold')
axes[0,1].set_xlabel('Betweenness Centrality')

# Top 10 by Eigenvector
top10_ev = centrality_df.nlargest(10, 'eigenvector')
axes[1,0].barh(range(len(top10_ev)), top10_ev['eigenvector'], color='purple', alpha=0.8)
axes[1,0].set_yticks(range(len(top10_ev)))
axes[1,0].set_yticklabels(top10_ev['user'], fontsize=8)
axes[1,0].invert_yaxis()
axes[1,0].set_title('Top 10 — Eigenvector\n(Connected to Important Users)', fontweight='bold')
axes[1,0].set_xlabel('Eigenvector Centrality')

# Community size distribution
top15_communities = community_sizes.head(15)
axes[1,1].bar(range(len(top15_communities)), top15_communities.values, color='green', alpha=0.7)
axes[1,1].set_title('Top 15 Community Sizes', fontweight='bold')
axes[1,1].set_xlabel('Community Rank')
axes[1,1].set_ylabel('Number of Users')

plt.suptitle('EV Reddit Network — Influence and Community Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'network_top_users.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Top users plot saved!')

## 12. Final Summary

In [ ]:
print('=' * 60)
print('NETWORK ANALYSIS COMPLETE')
print('=' * 60)
print(f'GRAPH STRUCTURE')
print(f'  Nodes (users)              : {N:,}')
print(f'  Edges (reply links)        : {M:,}')
print(f'  Density                    : {density:.6f}')
print(f'  Transitivity               : {transitivity:.4f}')
print(f'  Reciprocity                : {reciprocity:.4f}')
print(f'  Weakly connected comps     : {num_wcc:,}')
print(f'  Strongly connected comps   : {num_scc:,}')
print(f'')
print(f'CENTRALITY')
print(f'  Most influential (PageRank): {centrality_df.iloc[0]["user"]}')
print(f'  Top PageRank score         : {centrality_df.iloc[0]["pagerank"]:.6f}')
print(f'  Top Betweenness user       : {centrality_df.nlargest(1,"betweenness")["user"].values[0]}')
print(f'  Top Eigenvector user       : {centrality_df.nlargest(1,"eigenvector")["user"].values[0]}')
print(f'')
print(f'COMMUNITIES')
print(f'  Communities detected       : {num_communities}')
print(f'  Modularity score           : {modularity:.4f}')
print(f'  Largest community          : {community_sizes.iloc[0]} users')
print(f'')
print(f'FILES SAVED')
print(f'  data/processed/centrality_scores.csv')
print(f'  data/processed/communities.csv')
print(f'  data/processed/ev_reply_graph.graphml')
print(f'  data/processed/mod_ev_reply_graph.graphml')
print(f'  reports/figures/centrality_distributions.png')
print(f'  reports/figures/network_graph.png')
print(f'  reports/figures/network_top_users.png')
print('=' * 60)